# 03 — Baseline (§7 del README)

| | |
|---|---|
| **Entrada** | Split estratificado de §6 (`notebooks/02_preprocessing.ipynb`) |
| **Modelos** | Referencia trivial (`DummyClassifier`) + regresión logística |
| **Métricas** | Accuracy, precision, recall, F1, ROC-AUC y PR-AUC |
| **Salida** | Resultados reproducibles para README §7 y §10 |

**Reglas:** sin resultados inventados; toda cifra se obtiene ejecutando este notebook con `.venv`. La decisión formal de la métrica principal queda para §9.

## 1. Estrategia del baseline

El baseline responde: **¿cuánto mejora un modelo simple frente a no aprender nada?**

| Nivel | Modelo | Rol |
|---|---|---|
| 0 | `DummyClassifier(strategy='prior')` | Piso de referencia: predice siempre `no` usando la frecuencia de clases |
| 1 | Regresión logística | Modelo simple, rápido, probabilístico e interpretable; usa el pipeline de §6 |

**Por qué regresión logística:** es el baseline estándar para clasificación binaria, permite leer la dirección de cada variable y sirve de vara para modelos más complejos (§8).

**Criterio de comparación:** la clase de interés es `yes` (11,3%). Se prioriza **PR-AUC** (resume precision/recall con desbalance) y **recall/F1 de `yes`**; el accuracy **no** decide porque el dummy ya obtiene ~88,7% sin aprender.

**Limitaciones de esta etapa:** una sola partición train/test, sin ajuste de hiperparámetros, umbral fijo en 0,5 y un modelo lineal. La validación cruzada y la métrica principal se definen en §9.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

from src.preprocessing import (
    RANDOM_STATE,
    build_preprocessor,
    clean_data,
    load_raw_data,
    split_features_target,
    train_test_split_data,
)

df = clean_data(load_raw_data())
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)

print(f"Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas")
print(f"Tasa base de yes en test: {y_test.mean() * 100:.2f}%")

Train: 32,940 filas | Test: 8,236 filas
Tasa base de yes en test: 11.27%


## 2. Referencia trivial (`DummyClassifier`)

`strategy='prior'` predice siempre la clase mayoritaria (`no`) y asigna a `yes` la probabilidad igual a su frecuencia (11,3%). Sirve para mostrar por qué el accuracy engaña y para fijar el piso de ROC-AUC y PR-AUC.

In [2]:
resultados = {}
predicciones = {}


def evaluar(nombre, estimador):
    estimador.fit(X_train, y_train)
    y_pred = estimador.predict(X_test)
    y_proba = estimador.predict_proba(X_test)[:, 1]
    metricas = {
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "PR-AUC": average_precision_score(y_test, y_proba),
    }
    return metricas, y_pred


resultados["Dummy (prior)"], predicciones["Dummy (prior)"] = evaluar(
    "Dummy (prior)",
    DummyClassifier(strategy="prior", random_state=RANDOM_STATE),
)

print(pd.DataFrame([resultados["Dummy (prior)"]]).set_index("Modelo").round(3).to_string())
print("\nMatriz de confusion (filas: real | columnas: predicho):")
print(pd.DataFrame(
    confusion_matrix(y_test, predicciones["Dummy (prior)"]),
    index=["real no", "real yes"],
    columns=["pred no", "pred yes"],
).to_string())
print("\nReporte por clase:")
print(classification_report(
    y_test, predicciones["Dummy (prior)"],
    target_names=["no", "yes"], zero_division=0,
))

               Accuracy  Precision  Recall   F1  ROC-AUC  PR-AUC
Modelo                                                          
Dummy (prior)     0.887        0.0     0.0  0.0      0.5   0.113

Matriz de confusion (filas: real | columnas: predicho):
          pred no  pred yes
real no      7308         0
real yes      928         0

Reporte por clase:
              precision    recall  f1-score   support

          no       0.89      1.00      0.94      7308
         yes       0.00      0.00      0.00       928

    accuracy                           0.89      8236
   macro avg       0.44      0.50      0.47      8236
weighted avg       0.79      0.89      0.83      8236



## 3. Baseline supervisado: regresión logística

Pipeline completo: preprocesamiento de §6 + `LogisticRegression` con regularización L2 por defecto (`C=1`), `max_iter=1000` y `random_state=42`. El preprocesador se ajusta dentro del pipeline, solo con `X_train`.

Se muestran los coeficientes de mayor magnitud como **señal descriptiva** (no causal): al estar estandarizadas las numéricas y ser One-Hot las categóricas, el signo indica la dirección de la asociación con `yes`.

In [3]:
pipeline_log = Pipeline(steps=[
    ("preprocessor", build_preprocessor()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

resultados["Regresion logistica"], predicciones["Regresion logistica"] = evaluar(
    "Regresion logistica", pipeline_log
)

print(pd.DataFrame([resultados["Regresion logistica"]]).set_index("Modelo").round(3).to_string())
print("\nMatriz de confusion (filas: real | columnas: predicho):")
print(pd.DataFrame(
    confusion_matrix(y_test, predicciones["Regresion logistica"]),
    index=["real no", "real yes"],
    columns=["pred no", "pred yes"],
).to_string())
print("\nReporte por clase:")
print(classification_report(
    y_test, predicciones["Regresion logistica"],
    target_names=["no", "yes"], zero_division=0,
))

nombres = pipeline_log.named_steps["preprocessor"].get_feature_names_out()
coeficientes = pd.Series(
    pipeline_log.named_steps["model"].coef_[0], index=nombres
)
top_coef = coeficientes.reindex(
    coeficientes.abs().sort_values(ascending=False).index
).head(12)

print("\nCoeficientes de mayor magnitud (features estandarizadas/One-Hot):")
print(top_coef.round(3).to_string())

                     Accuracy  Precision  Recall     F1  ROC-AUC  PR-AUC
Modelo                                                                  
Regresion logistica     0.898      0.693   0.168  0.271    0.776    0.43

Matriz de confusion (filas: real | columnas: predicho):
          pred no  pred yes
real no      7239        69
real yes      772       156

Reporte por clase:
              precision    recall  f1-score   support

          no       0.90      0.99      0.95      7308
         yes       0.69      0.17      0.27       928

    accuracy                           0.90      8236
   macro avg       0.80      0.58      0.61      8236
weighted avg       0.88      0.90      0.87      8236


Coeficientes de mayor magnitud (features estandarizadas/One-Hot):
poutcome_failure       -1.077
emp.var.rate           -0.793
poutcome_success        0.779
poutcome_nonexistent   -0.518
nr.employed            -0.373
marital_divorced       -0.359
job_retired             0.306
cons.price.idx  

## 4. Efecto del desbalance

En este problema, un **falso negativo** (no llamar a quien habría contratado) cuesta más que un **falso positivo** (llamada desperdiciada). Por eso se prueba `class_weight='balanced'`, que pondera las clases inversamente a su frecuencia. No se remuestrea: así se evita sintetizar datos y se mantiene el pipeline simple.

In [4]:
pipeline_log_bal = Pipeline(steps=[
    ("preprocessor", build_preprocessor()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

resultados["Regresion logistica balanceada"], predicciones["Regresion logistica balanceada"] = evaluar(
    "Regresion logistica balanceada", pipeline_log_bal
)

print(pd.DataFrame([resultados["Regresion logistica balanceada"]]).set_index("Modelo").round(3).to_string())
print("\nMatriz de confusion (filas: real | columnas: predicho):")
print(pd.DataFrame(
    confusion_matrix(y_test, predicciones["Regresion logistica balanceada"]),
    index=["real no", "real yes"],
    columns=["pred no", "pred yes"],
).to_string())
print("\nReporte por clase:")
print(classification_report(
    y_test, predicciones["Regresion logistica balanceada"],
    target_names=["no", "yes"], zero_division=0,
))

tabla = pd.DataFrame(resultados.values()).set_index("Modelo").round(3)
print("\nComparacion de los tres escenarios:")
print(tabla.to_string())

delta = (tabla.loc["Regresion logistica balanceada"] - tabla.loc["Regresion logistica"]).round(3)
print("\nCambio al usar class_weight='balanced':")
print(delta.to_string())

                                Accuracy  Precision  Recall    F1  ROC-AUC  PR-AUC
Modelo                                                                            
Regresion logistica balanceada     0.773       0.29   0.699  0.41    0.777   0.429

Matriz de confusion (filas: real | columnas: predicho):
          pred no  pred yes
real no      5719      1589
real yes      279       649

Reporte por clase:
              precision    recall  f1-score   support

          no       0.95      0.78      0.86      7308
         yes       0.29      0.70      0.41       928

    accuracy                           0.77      8236
   macro avg       0.62      0.74      0.63      8236
weighted avg       0.88      0.77      0.81      8236


Comparacion de los tres escenarios:
                                Accuracy  Precision  Recall     F1  ROC-AUC  PR-AUC
Modelo                                                                             
Dummy (prior)                      0.887      0.000   0.00

## 5. Comparación y lectura

La tabla anterior es la evidencia de §7. Para leerla:

- Frente al dummy, el modelo debe subir **PR-AUC** y **recall/F1 de `yes`**; si solo sube accuracy, no está aprendiendo lo relevante.
- `class_weight='balanced'` suele **subir recall** y bajar precision: se acepta solo si la ganancia en recall/PR-AUC compensa las llamadas extra.
- La decisión de adoptar o no el balanceo se documenta en el README §7 con estas cifras; la métrica principal formal se fija en §9.

## 6. Resumen de §7 y próximos pasos

| Punto del README §7 | Contenido |
|---|---|
| Estrategia/modelo | Dummy `prior` (referencia) + regresión logística L2 |
| Métricas | Accuracy, precision, recall, F1, ROC-AUC y PR-AUC sobre el test holdout |
| Criterio | PR-AUC y recall/F1 de `yes`; accuracy solo como control |
| Limitaciones | Un solo split, sin tuning, umbral 0,5, modelo lineal |

**Próximo paso:** §8 — seleccionar y evaluar modelos adicionales (árboles/ensambles) bajo una estrategia común; §9 — validación cruzada, métrica principal y umbral.